# 05 — Monitoring pack / report

**Plain English:** We pull the key tables together into a short, disclosure-style monitoring pack and write it to `outputs/reports/report.md`.

Includes a 3-stage **IFRS 9-style proxy** (Stage 1 performing / Stage 2 problem-exposure / Stage 3 charged-off) and a credit-quality table laid out in **APS 330-style disclosure format**.

> ⚠️ **Labelling:** these are *monitoring outputs on public SBA data*, not a regulated entity's disclosure. The APS 330 layout is used for familiarity only. Full IFRS 9 staging and transition matrices live in the companion Freddie Mac monitor.

In [1]:
%matplotlib inline
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option('display.max_columns', 40, 'display.width', 200)
from src.config import load_config, TABLES_DIR
CFG = load_config()

In [2]:
from src.data_loader import load_clean
from src.base_table import build_base_table
loans = load_clean(config=CFG)
base = build_base_table(loans, config=CFG)
print(f'{len(base):,} funded loans, vintages {int(base.vintage.min())}-{int(base.vintage.max())}')

[2026-06-25 13:11:20] [INFO    ] [data_loader] Loading 2 SBA input file(s): ['foia-7a-fy2000-fy2009-asof-260331.csv', 'foia-7a-fy2010-fy2019-asof-260331.csv']


[2026-06-25 13:11:23] [INFO    ] [data_loader]   foia-7a-fy2000-fy2009-asof-260331.csv -> 690333 rows


[2026-06-25 13:11:26] [INFO    ] [data_loader]   foia-7a-fy2010-fy2019-asof-260331.csv -> 545751 rows


[2026-06-25 13:11:26] [INFO    ] [data_loader] Combined raw frame: 1236084 rows


[2026-06-25 13:11:29] [INFO    ] [data_loader] Dropped 149065 never-funded loans (['CANCLD', 'COMMIT']); 1087019 funded loans remain


[2026-06-25 13:11:31] [INFO    ] [base_table] Base table built: 1087019 loans, 20 vintages (2000-2019), 132662 defaults, 9805 problem exposures


1,087,019 funded loans, vintages 2000-2019


In [3]:
from src import report as rpt, concentration as conc, chargeoff as co
from src.data_loader import load_clean, data_quality_summary
from src.early_warning import flag_high_risk_segments
dq = data_quality_summary(base, config=CFG)
stage = rpt.stage_proxy_summary(base, config=CFG)
aps = rpt.aps330_style_credit_quality(base)
stage.to_csv(TABLES_DIR / '05_stage_proxy_summary.csv', index=False)
aps.to_csv(TABLES_DIR / '05_aps330_style_credit_quality.csv', index=False)

### Result: 3-stage IFRS 9-style proxy (Stage 1 / 2 / 3)

In [4]:
stage

,proxy_stage,loan_count,exposure,chargeoff_amount,exposure_share
0,Stage 1 — performing,944552,2.591538e+11,5.611564e+09,0.9004
1,Stage 2 — problem exposure (SICR proxy),9805,6.523461e+09,4.753578e+05,0.0227
2,Stage 3 — charged off (default),132662,2.213175e+10,1.416626e+10,0.0769


#### APS 330-style credit-quality by industry *(format only — not a regulatory disclosure)*

In [5]:
aps.head(12)

,industry,gross_exposure,loan_count,charged_off_count,charged_off_amount,exposure_share,impairment_rate_dollar
0,Accommodation & Food Services,5.101697e+10,133403,16373,3.659654e+09,0.1773,0.0717
1,Retail Trade,4.418474e+10,174624,24874,3.639054e+09,0.1535,0.0824
2,Manufacturing,3.030025e+10,87762,9328,1.945051e+09,0.1053,0.0642
3,Health Care & Social Assistance,2.959416e+10,87527,5924,1.248399e+09,0.1028,0.0422
4,Other Services (except Public Admin),2.427371e+10,104527,12883,1.791361e+09,0.0843,0.0738
5,"Professional, Scientific & Technical Services",2.038662e+10,104993,11936,1.209750e+09,0.0708,0.0593
6,Wholesale Trade,1.856048e+10,60185,8907,1.192667e+09,0.0645,0.0643
7,Construction,1.707175e+10,104884,14342,1.570478e+09,0.0593,0.0920
8,"Arts, Entertainment & Recreation",8.271858e+09,26375,3307,6.360027e+08,0.0287,0.0769
9,Administrative & Waste Management Services,8.128158e+09,53337,6916,6.538930e+08,0.0282,0.0804


#### Assemble and write the full Markdown report

In [6]:
md_report = rpt.build_markdown_report(
    dq=dq, hhi=conc.hhi_summary(base),
    co_industry=co.chargeoff_by_industry(base),
    co_vintage=co.chargeoff_by_vintage(base),
    early_warning=flag_high_risk_segments(base, config=CFG),
    stage_proxy=stage,
)
(ROOT / 'outputs' / 'reports').mkdir(parents=True, exist_ok=True)
(ROOT / 'outputs' / 'reports' / 'report.md').write_text(md_report, encoding='utf-8')
print(md_report[:1200])

[2026-06-25 13:11:32] [INFO    ] [concentration] Concentration industry: HHI=0.1004 (Moderate), 21 segments


[2026-06-25 13:11:33] [INFO    ] [concentration] Concentration state: HHI=0.0577 (Low), 60 segments


[2026-06-25 13:11:33] [INFO    ] [concentration] Concentration lender: HHI=0.0144 (Low), 4222 segments


[2026-06-25 13:11:35] [INFO    ] [concentration] Concentration borrower: HHI=0.0000 (Low), 910400 segments


[2026-06-25 13:11:35] [INFO    ] [early_warning] Early warning: 115 segments flagged (portfolio rate 13.46%, >= 1.5x threshold, min 200 loans)


# Commercial Portfolio Monitoring Pack — SBA 7(a)

_Real public SBA 7(a) loan-level data. Monitoring outputs only — not a regulated disclosure. Any APS 330-style table below is laid out in that format for familiarity and is labelled accordingly._

## 1. Portfolio at a glance

| Metric | Value |
|---|---|
| Funded loans | 1,087,019 |
| Approval FY range | 2000-2019 |
| Total gross approval ($) | 287,809,010,585 |
| Charged-off loans (count) | 132,662 |
| Charge-off rate (count) | 12.20% |
| Charge-off rate ($) | 4.92% |
| Non-performing loans (NPL proxy, count) | 142,467 |
| Non-performing rate (NPL proxy, count) | 13.11% |
| Distinct NAICS sectors | 21 |
| Distinct borrower states | 60 |
|   of which 50 states + DC | 1,075,335 loans |
|   of which territories / military / FAS | 11,684 loans |
|   of which unrecognised state code | 0 loans |
| Distinct borrowers (names) | 910,399 |
| Distinct lenders (named) | 4,221 |
| Missing lender name | 27 |
| Missing borrower name | 23 |
| Missing

**Read-out:** `outputs/reports/report.md` is the one-page pack a credit committee would skim — portfolio size, concentration, worst industries/vintages, flagged segments, and the Stage 1/2/3 split.